## Imports

In [25]:
from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import VectorAssembler

In [2]:
# dataset - https://www.kaggle.com/code/krantiswalke/bank-personal-loan-modelling-supervised-learning/input

In [3]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/Cellar/openjdk@17/17.0.16/libexec/openjdk.jdk/Contents/Home"  # <-- replace with your exact output
os.environ["SHELL"] = "/bin/bash"
os.environ["PATH"] = (
    os.path.join(os.environ["JAVA_HOME"], "bin") +
    ":/usr/local/bin:/usr/bin:/bin:/usr/sbin:/sbin"
)


In [4]:
spark = SparkSession \
    .builder \
    .appName("Visa for Lisa") \
    .getOrCreate()

spark.conf.set("spark.sql.debug.maxToStringFields", 1000)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/19 14:27:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Data Loading

In [5]:
df = spark.read \
    .option("header", True) \
    .csv("Visa_For_Lisa_Loan_Modelling.csv", inferSchema=True)

In [6]:
df.show()

+---+---+----------+------+--------+------+-----+---------+--------+-------------+------------------+----------+------+----------+
| ID|Age|Experience|Income|ZIP Code|Family|CCAvg|Education|Mortgage|Personal Loan|Securities Account|CD Account|Online|CreditCard|
+---+---+----------+------+--------+------+-----+---------+--------+-------------+------------------+----------+------+----------+
|  1| 25|         1|    49|   91107|     4|  1.6|        1|       0|            0|                 1|         0|     0|         0|
|  2| 45|        19|    34|   90089|     3|  1.5|        1|       0|            0|                 1|         0|     0|         0|
|  3| 39|        15|    11|   94720|     1|  1.0|        1|       0|            0|                 0|         0|     0|         0|
|  4| 35|         9|   100|   94112|     1|  2.7|        2|       0|            0|                 0|         0|     0|         0|
|  5| 35|         8|    45|   91330|     4|  1.0|        2|       0|            0| 

In [7]:
df.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- Income: integer (nullable = true)
 |-- ZIP Code: integer (nullable = true)
 |-- Family: integer (nullable = true)
 |-- CCAvg: double (nullable = true)
 |-- Education: integer (nullable = true)
 |-- Mortgage: integer (nullable = true)
 |-- Personal Loan: integer (nullable = true)
 |-- Securities Account: integer (nullable = true)
 |-- CD Account: integer (nullable = true)
 |-- Online: integer (nullable = true)
 |-- CreditCard: integer (nullable = true)



In [8]:
df.describe().toPandas()

,summary,ID,Age,Experience,Income,ZIP Code,Family,CCAvg,Education,Mortgage,Personal Loan,Securities Account,CD Account,Online,CreditCard
0,count,5000,5000,5000,5000,5000,5000,5000,5000,5000,5000,5000,5000,5000,5000
1,mean,2500.5,45.3384,20.1046,73.7742,93152.503,2.3964,1.9379380000000053,1.881,56.4988,0.096,0.1044,0.0604,0.5968,0.294
2,stddev,1443.5200033252052,11.463165630542662,11.46795368112056,46.03372932108627,2121.8521973361953,1.1476630455378507,1.7476589800467675,0.839869082664199,101.71380210211213,0.29462070577617994,0.30580932600032634,0.23825027311322794,0.4905893349626712,0.4556374886949281
3,min,1,23,-3,8,9307,1,0.0,1,0,0,0,0,0,0
4,max,5000,67,43,224,96651,4,10.0,3,635,1,1,1,1,1


In [9]:
df.groupBy("CD Account").count().show()

+----------+-----+
|CD Account|count|
+----------+-----+
|         1|  302|
|         0| 4698|
+----------+-----+



In [22]:
deposits = df.filter("`CD Account` = 1")

In [23]:
deposits.show()

+---+---+----------+------+--------+------+-----+---------+--------+-------------+------------------+----------+------+----------+
| ID|Age|Experience|Income|ZIP Code|Family|CCAvg|Education|Mortgage|Personal Loan|Securities Account|CD Account|Online|CreditCard|
+---+---+----------+------+--------+------+-----+---------+--------+-------------+------------------+----------+------+----------+
| 30| 38|        13|   119|   94104|     1|  3.3|        2|       0|            1|                 0|         1|     1|         1|
| 39| 42|        18|   141|   94114|     3|  5.0|        3|       0|            1|                 1|         1|     1|         0|
| 48| 37|        12|   194|   91380|     4|  0.2|        3|     211|            1|                 1|         1|     1|         1|
| 57| 55|        30|    29|   94005|     3|  0.1|        2|       0|            0|                 1|         1|     1|         0|
| 76| 31|         7|   135|   94901|     4|  3.8|        2|       0|            1| 

In [24]:
deposits.groupBy("Personal Loan").count().show()

+-------------+-----+
|Personal Loan|count|
+-------------+-----+
|            1|  140|
|            0|  162|
+-------------+-----+



In [27]:
feature_assembler = VectorAssembler(
    inputCols=[
        "Age", "Experience", "Income", "ZIP Code", "Family", "CCAvg", 
        "Education", "Mortgage", "Securities Account", "Online", "CreditCard"
    ],
    outputCol="independent_features"
    )

In [58]:
dataset = feature_assembler.transform(deposits)
dataset.show()

+---+---+----------+------+--------+------+-----+---------+--------+-------------+------------------+----------+------+----------+--------------------+
| ID|Age|Experience|Income|ZIP Code|Family|CCAvg|Education|Mortgage|Personal Loan|Securities Account|CD Account|Online|CreditCard|independent_features|
+---+---+----------+------+--------+------+-----+---------+--------+-------------+------------------+----------+------+----------+--------------------+
| 30| 38|        13|   119|   94104|     1|  3.3|        2|       0|            1|                 0|         1|     1|         1|[38.0,13.0,119.0,...|
| 39| 42|        18|   141|   94114|     3|  5.0|        3|       0|            1|                 1|         1|     1|         0|[42.0,18.0,141.0,...|
| 48| 37|        12|   194|   91380|     4|  0.2|        3|     211|            1|                 1|         1|     1|         1|[37.0,12.0,194.0,...|
| 57| 55|        30|    29|   94005|     3|  0.1|        2|       0|            0|      

In [59]:
dataset = dataset.select(["ID", "Personal Loan", "independent_features"])
dataset.show()

+---+-------------+--------------------+
| ID|Personal Loan|independent_features|
+---+-------------+--------------------+
| 30|            1|[38.0,13.0,119.0,...|
| 39|            1|[42.0,18.0,141.0,...|
| 48|            1|[37.0,12.0,194.0,...|
| 57|            0|[55.0,30.0,29.0,9...|
| 76|            1|[31.0,7.0,135.0,9...|
|132|            1|[58.0,34.0,149.0,...|
|139|            0|[59.0,34.0,42.0,9...|
|151|            0|[46.0,22.0,118.0,...|
|154|            0|[60.0,36.0,22.0,9...|
|200|            1|[36.0,11.0,158.0,...|
|210|            1|[64.0,39.0,172.0,...|
|228|            0|[47.0,23.0,148.0,...|
|229|            0|[47.0,22.0,53.0,9...|
|244|            1|[65.0,39.0,170.0,...|
|248|            1|[53.0,29.0,120.0,...|
|263|            0|[49.0,23.0,33.0,9...|
|289|            1|[44.0,19.0,172.0,...|
|300|            1|[41.0,15.0,159.0,...|
|309|            0|[32.0,8.0,128.0,9...|
|323|            1|[63.0,39.0,101.0,...|
+---+-------------+--------------------+
only showing top

In [60]:
train_data, test_data = dataset.randomSplit([0.75, 0.25])

In [61]:
regressor = LinearRegression(featuresCol="independent_features", labelCol="Personal Loan")
lr_model = regressor.fit(train_data)

25/08/19 15:09:02 WARN Instrumentation: [ecd2d0ad] regParam is zero, which might cause numerical instability and overfitting.


In [62]:
lr_model.coefficients

DenseVector([0.0176, -0.0153, 0.0061, -0.0, 0.0601, -0.0066, 0.0863, -0.0001, -0.1258, -0.3113, -0.2918])

In [63]:
lr_model.intercept

0.9896944234829792

In [64]:
pred_results = lr_model.evaluate(test_data)
pred_results.predictions.show()

+----+-------------+--------------------+--------------------+
|  ID|Personal Loan|independent_features|          prediction|
+----+-------------+--------------------+--------------------+
|  76|            1|[31.0,7.0,135.0,9...|  0.6381505837468217|
| 154|            0|[60.0,36.0,22.0,9...| -0.2903193496720322|
| 263|            0|[49.0,23.0,33.0,9...| 0.08804848316877922|
| 323|            1|[63.0,39.0,101.0,...| 0.44569806927104993|
| 327|            0|[52.0,27.0,80.0,9...| 0.27932147400061724|
| 349|            1|[40.0,15.0,173.0,...|  0.7972926234140235|
| 406|            0|[36.0,11.0,133.0,...|  0.2993845814505386|
| 439|            1|[58.0,32.0,113.0,...|  0.4783696870534849|
| 556|            0|[34.0,8.0,35.0,92...|0.030590405761439787|
| 716|            0|[47.0,23.0,32.0,9...| -0.2837750639012093|
| 744|            0|[61.0,37.0,40.0,9...|-0.06631281095112374|
| 755|            0|[38.0,14.0,102.0,...|  0.2580916993368999|
| 783|            1|[54.0,30.0,194.0,...|  0.9169674041

In [65]:
pred_results.r2, pred_results.meanAbsoluteError, pred_results.meanSquaredError

(0.5720825418791231, 0.2658108736524642, 0.1035866993762303)

In [68]:
pred_results.predictions.select(["ID", "Personal Loan", "prediction"]).write.csv("predicted_results_1.csv", header=True, mode="overwrite")

In [78]:
pred_results.predictions.plot.scatter(y=["Personal Loan", "prediction"], x="ID")

In [77]:
pred_results.predictions.plot.line(y=["Personal Loan", "prediction"], x="ID")